# GeoCalib-Align Colab Notebook

End-to-end Colab workflow for free, open-source fine-tuning and evaluation.

## Section 0: Workspace Setup

This notebook assumes the repository files are already available in Colab, either by cloning the repository into `/content/geocalib_align` or by uploading the project folder.

In [ ]:
from pathlib import Path
import os

REPO_URL = os.environ.get("GEOCALIB_ALIGN_REPO_URL", "https://github.com/elom354/geocalib_align.git")
REPO_DIR = Path("/content/GEOCALIB_ALIGN")

if REPO_DIR.exists():
    print("Repo already present at", REPO_DIR)
else:
    !git clone "$REPO_URL" "$REPO_DIR"

%cd /content/GEOCALIB_ALIGN
print("Repo ready at", REPO_DIR)

## Section 1: Environment Setup

Dependencies are installed in a Colab-friendly way without downgrading core system packages such as NumPy, SciPy, Torch, or `fsspec`. This avoids binary incompatibility errors and reduces resolver conflicts with the preloaded Colab stack.

In [ ]:
%%time
from pathlib import Path
import importlib.metadata as importlib_metadata
import subprocess
import sys

sentinel = Path('/tmp/geocalib_align_colab_deps_v3')
required = {
    'transformers': '4.46.3',
    'peft': '0.13.2',
    'accelerate': '0.34.2',
    'trl': '0.10.1',
    'bert-score': '0.3.13',
    'kaleido': '0.2.1',
}

def installed_version(dist_name):
    try:
        return importlib_metadata.version(dist_name)
    except importlib_metadata.PackageNotFoundError:
        return None

to_install = []
for dist_name, expected_version in required.items():
    current_version = installed_version(dist_name)
    if current_version != expected_version:
        to_install.append(f'{dist_name}=={expected_version}')

if to_install and not sentinel.exists():
    print({'installing': to_install})
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade-strategy', 'only-if-needed', *to_install]
    subprocess.run(cmd, check=True)
    sentinel.write_text('installed')
    raise SystemExit('Packages were installed. Restart the Colab session manually, then rerun the notebook from the top.')
elif to_install and sentinel.exists():
    raise RuntimeError('Required package versions are still not active after installation. Use Runtime > Factory reset runtime, then rerun from the top.')
else:
    print('Dependency layer is ready.')


In [ ]:
import os
from getpass import getpass
import torch

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'
gpu_vram = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if torch.cuda.is_available() else 0
print({'gpu_available': torch.cuda.is_available(), 'gpu_name': gpu_name, 'gpu_vram_gb': gpu_vram})
os.environ['HF_TOKEN'] = getpass('Hugging Face token (optional, leave blank for public models): ')

## Section 2: Data Loading

GeoBench, EarthSE, and GeoSignal are downloaded automatically by the repository scripts using verified dataset IDs.

In [ ]:
%%time
import json
import subprocess
import sys
from pathlib import Path
import matplotlib.pyplot as plt

for script in ['data/download_data.py', 'data/prepare_geosignal.py', 'data/build_physgeo_eval.py']:
    subprocess.run([sys.executable, script], check=True)

train_path = Path('data/processed/geosignal_train.json')
if not train_path.exists():
    raise FileNotFoundError('GeoSignal preparation did not create data/processed/geosignal_train.json')

train_records = json.loads(train_path.read_text())
for sample in train_records[:3]:
    print(sample)

lengths = [len((row['instruction'] + ' ' + row['input'] + ' ' + row['output']).split()) for row in train_records]
plt.hist(lengths, bins=30)
plt.title('GeoSignal Token Length Distribution')
plt.xlabel('Approximate word count')
plt.ylabel('Frequency')
plt.show()


## Section 3: Model Loading

A public non-gated model is used by default so the notebook stays free and does not require special access approval.

## Section 3A: Colab Demo Config

Create a smaller Colab-specific config to keep the demo stable on a free T4 runtime.

In [ ]:
import yaml
from pathlib import Path

base_config = Path('config/experiments.yaml')
demo_config = Path('config/experiments_colab_demo.yaml')
cfg = yaml.safe_load(base_config.read_text())
cfg['finetuning']['max_steps'] = 30
cfg['finetuning']['warmup_steps'] = 5
cfg['finetuning']['batch_size'] = 1
cfg['finetuning']['gradient_accumulation'] = 2
cfg['finetuning']['max_seq_length'] = 512
cfg['finetuning']['load_in_4bit'] = False
demo_config.write_text(yaml.safe_dump(cfg, sort_keys=False))
print({'demo_config': str(demo_config)})


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
    low_cpu_mem_usage=True,
)
print(model.__class__.__name__)
print('Trainable parameters before LoRA:', sum(p.numel() for p in model.parameters() if p.requires_grad))


## Section 4: Fine-Tuning (Standard LoRA)

In [ ]:
%%time
!python finetune/train_lora_standard.py --model_id Qwen/Qwen2.5-1.5B-Instruct --output_dir results/colab_demo/lora_standard --config config/experiments_colab_demo.yaml

## Section 5: Fine-Tuning (Selective LoRA)

In [ ]:
%%time
!python finetune/train_lora_selective.py --model_id Qwen/Qwen2.5-1.5B-Instruct --output_dir results/colab_demo/lora_selective --config config/experiments_colab_demo.yaml

## Section 6: Fine-Tuning (LoRA + Replay)

In [ ]:
%%time
!python finetune/train_lora_replay.py --model_id Qwen/Qwen2.5-1.5B-Instruct --output_dir results/colab_demo/lora_replay --config config/experiments_colab_demo.yaml

## Section 7: Fine-Tuning (Mix-CPT)

In [ ]:
%%time
!python finetune/train_mix_cpt.py --model_id Qwen/Qwen2.5-1.5B-Instruct --output_dir results/colab_demo/mix_cpt --config config/experiments_colab_demo.yaml

## Section 8: Evaluation

The free evaluation stack uses local open-source models only. PhysGeo evaluation is skipped automatically until the annotation template has been filled with real expert-reviewed examples.

In [ ]:
%%time
import json
import subprocess
import sys
from pathlib import Path
import pandas as pd

checkpoint_dir = Path('results/colab_demo/lora_standard')
if not checkpoint_dir.exists():
    raise FileNotFoundError('Checkpoint results/colab_demo/lora_standard not found. Run the Standard LoRA training cell successfully before evaluation.')

closed_cmd = [sys.executable, 'evaluate/eval_closed_tasks.py', '--model_path', str(checkpoint_dir), '--model_name', 'Qwen2.5-1.5B', '--strategy', 'lora_std', '--max_samples', '100']
open_cmd = [sys.executable, 'evaluate/eval_open_tasks.py', '--model_path', str(checkpoint_dir), '--model_name', 'Qwen2.5-1.5B', '--strategy', 'lora_std', '--judge_model', 'Qwen/Qwen2.5-1.5B-Instruct', '--max_samples', '40']
subprocess.run(closed_cmd, check=True)
subprocess.run(open_cmd, check=True)

template = json.loads(Path('data/physgeo_eval_template.json').read_text())
if template.get('examples'):
    subprocess.run([
        sys.executable, 'evaluate/eval_physgeo.py', '--model_path', str(checkpoint_dir), '--model_name', 'Qwen2.5-1.5B', '--strategy', 'lora_std', '--judge_model', 'Qwen/Qwen2.5-1.5B-Instruct'
    ], check=True)
    subprocess.run([sys.executable, 'evaluate/aggregate_results.py'], check=True)
    display(pd.read_csv('results/summary.csv').head())
else:
    print('PhysGeo template is empty by design. Add expert-reviewed examples to data/physgeo_eval_template.json, then rerun PhysGeo evaluation and aggregation.')
    open_results = json.loads(Path('results/Qwen2.5-1.5B/lora_std/open_results.json').read_text())
    closed_results = json.loads(Path('results/Qwen2.5-1.5B/lora_std/closed_results.json').read_text())
    print({'closed_overall': closed_results['overall_accuracy'], 'prompt_alignment': open_results['prompt_alignment'], 'correctness': open_results['correctness'], 'answer_relevance': open_results['answer_relevance']})


## Section 9: Figure Generation

Figures are generated only when `results/summary.csv` exists, which requires closed, open, and PhysGeo evaluation outputs.

In [ ]:
%%time
from pathlib import Path
import subprocess
import sys
from IPython.display import Image, display

summary_path = Path('results/summary.csv')
if summary_path.exists():
    subprocess.run([sys.executable, 'figures/generate_all_figures.py', '--data', str(summary_path)], check=True)
    figure_names = [
        'fig1_tradeoff_scatter',
        'fig2_radar_chart',
        'fig3_heatmap',
        'fig4_bar_strategies',
        'fig5_physgeo_breakdown',
        'fig6_cost_performance',
        'fig7_alignment_delta',
        'fig8_leaderboard',
    ]
    for figure_name in figure_names:
        display(Image(filename=f'figures/{figure_name}.png'))
else:
    print('Skipping figure generation because results/summary.csv does not exist yet.')

## Section 10: Export

In [ ]:
%%time
from pathlib import Path
from google.colab import files

summary_path = Path('results/summary.csv')
if summary_path.exists():
    !zip -r geocalib_align_figures.zip figures/*.pdf figures/*.png results/summary.csv
    files.download('results/summary.csv')
    files.download('geocalib_align_figures.zip')
else:
    print('Skipping export because results/summary.csv does not exist yet.')